In [0]:
from pyspark.sql import functions as F

RUTA = "/Volumes/dunnhumby/bronze/raw_files"

# Tablas incluidas en el proyecto (causal_data se excluye por tamaño)
TABLAS = [
    "campaign_desc", "campaign_table", "coupon", "coupon_redempt",
    "hh_demographic", "product", "transaction_data",   # la más pesada va al final
]

def cargar_bronze(nombre):
    """Lee un CSV del Volume y lo guarda como tabla Delta en bronze."""
    df = (
        spark.read
        .option("header", True)   # sin inferSchema: todo entra como texto, los tipos se definen en Silver
        .csv(f"{RUTA}/{nombre}.csv")
        .select("*", F.col("_metadata.file_path").alias("_archivo_origen"))  # auditoría: de qué archivo vino cada fila
        .withColumn("_fecha_carga", F.current_timestamp())                   # auditoría: cuándo se cargó
    )
    (df.write
       .format("delta")
       .mode("overwrite")   # permite re-ejecutar el proceso sin duplicar datos
       .saveAsTable(f"dunnhumby.bronze.{nombre}"))
    # El conteo se hace sobre la tabla Delta guardada, no sobre el CSV, para no leer el archivo dos veces
    print(f"{nombre}: {spark.table(f'dunnhumby.bronze.{nombre}').count():,} filas")

for tabla in TABLAS:
    cargar_bronze(tabla)

In [0]:
display(spark.table("dunnhumby.bronze.campaign_desc"))